In [1]:
%%capture
%matplotlib inline
from collections import Counter
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "statnlpbook").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from statnlpbook.lm import NGramLM, OOV, UniformLM, perplexity, replace_OOVs

TRAIN_SENTENCES = [
    "the students opened their notebooks",
    "the students opened their laptops",
    "the students read their notes",
    "the students studied language models",
    "the students compared language models",
    "the lecture introduced language models",
    "the lecture explained word probabilities",
    "the model predicts the next word",
    "the model assigns probabilities to words",
    "language models predict words",
    "language models score sentences",
    "n gram models count short contexts",
    "n gram models use fixed contexts",
    "neural models learn context representations",
    "recurrent networks model longer contexts",
    "transformers use attention over tokens",
    "the rain started after the lecture",
    "the students opened their umbrellas",
    "the students left the classroom",
    "the class discussed word embeddings",
]
TEST_SENTENCES = [
    "the students opened their notes",
    "language models assign probabilities to words",
    "the model predicts a new token",
]

def tokenise(sentences):
    return [
        token
        for sentence in sentences
        for token in f"{sentence.lower()} .".split()
    ]

# Repetition gives the toy corpus stable counts. The final sentence supplies rare words.
train = tokenise(TRAIN_SENTENCES * 2 + ["the students encountered zeugma"])
test = tokenise(TEST_SENTENCES)
counts = Counter(train)
sorted_counts = sorted(counts.values(), reverse=True)
ranks = range(1, len(sorted_counts) + 1)

def top_probabilities(lm, history=(), how_many=8):
    pairs = sorted(
        ((word, lm.probability(word, *history)) for word in lm.vocab),
        key=lambda item: (-item[1], item[0]),
    )
    return [(word, round(probability, 3)) for word, probability in pairs[:how_many]]

def plot_probabilities(lm, context=(), how_many=8):
    pairs = top_probabilities(lm, context, how_many)
    plt.figure(figsize=(8, 4))
    plt.bar([word for word, _ in pairs], [probability for _, probability in pairs])
    plt.ylabel("probability")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    return pairs

<!-- No custom LaTeX macros: exported slides use standard MathJax commands. -->

In [ ]:
from IPython.display import Image
import random

In [ ]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>

# Language Modelling


## Learning objectives

By the end of this lecture, you should be able to:

- Explain language modelling as next-token prediction
- Estimate a simple n-gram model and explain what smoothing changes
- Interpret perplexity and state when comparisons are valid
- Relate count-based language models and static word embeddings to the RNN and Transformer models introduced in the next two lectures

## Language models predict what comes next

A language model assigns probabilities to possible next tokens and, from them, to whole sequences.

> The students opened their ...

Which continuation is most likely?

> The students opened their **notebooks / laptops / rainfall**

A different context changes the distribution.

> After the downpour, the students opened their **umbrellas**

A language model also scores complete sequences.

> The students opened their notebooks.

Which sequence should receive higher probability? Why?

> The students opened their rainfall.

## Where language models are used

- autocomplete and text generation
- scoring alternatives in speech recognition and machine translation
- pre-training representations that can be adapted to downstream tasks

The model family changes across the course, but the next-token objective remains central.

## Where this lecture fits

**Today:** n-gram models use explicit counts and a fixed context window. Static embeddings give each word one learned vector.

**Next week:** RNNs learn a representation of the preceding context and support neural language models.

**Week 40:** attention and Transformers produce contextual representations and power modern pretrained language models.

## Why begin with n-grams?

- We can inspect every count and probability.
- They provide fast baselines for more complex systems.
- Their failures reveal the problems that neural models address: sparse data and a fixed, short history.

We will learn the core ideas from a unigram and a bigram, then move on.

... but first, the basics

## Overview

* Language Modelling from scratch
* Evaluation
* Training
* Smoothing

## Probability of a sequence

A language model assigns a probability to a token sequence:

$$p(w_1,\ldots,w_d)$$ 



## Sequence probability and the chain rule

Any sequence probability can be factored into next-token probabilities:

\begin{align}
p(w_1,\ldots,w_d) &= p(w_1) p(w_2|w_1) p(w_3|w_1, w_2) \ldots \\
 &= p(w_1) \prod_{i = 2}^d p(w_i|w_1,\ldots,w_{i-1})
\end{align}

### Structured Prediction

predict word $y=w_i$ 
* conditioned on history $\mathbf{x}=w_1,\ldots,w_{i-1}$.

## The n-gram approximation

Estimating a separate probability for every complete history is impossible in realistic data. Most histories occur once or never.

$$
\mathbf{x}=w_1,\ldots,w_{i-1}
$$

### A fixed context window

Keep only the last $n-1$ words of the history:

$$
\mathbf{f}(\mathbf{x})=w_{i-(n-1)},\ldots,w_{i-1}
$$

$p(\text{notebooks}|\text{...,the students opened their}) 
= p(\text{notebooks}|\text{opened their})$

### Unigram LM

Set $n=1$:
$$
p(w_i|w_1,\ldots,w_{i-1}) = p(w_i).
$$

$p(\text{notebooks}|\text{the students opened their}) = p(\text{notebooks})$

### Bigram approximation

Set $n=2$:
$$
p(w_i|w_1,\ldots,w_{i-1}) = p(w_i|w_{i-1}).
$$

$p(\text{notebooks}|\text{the students opened their}) = p(\text{notebooks}|\text{their})$

### *Uniform* LM
Set $n=0$:

Same probability for each word in the vocabulary \\(V\\):

$$
p(w_i|w_1,\ldots,w_{i-1}) = \frac{1}{|V|}.
$$

$p(\text{notebooks}) = p(\text{umbrellas}) = \frac{1}{|V|}$

Let us look at a training set and create a uniform LM from it.

In [ ]:
train[:9]

In [ ]:
vocab = set(train)
baseline = UniformLM(vocab)
sum([baseline.probability(w) for w in vocab])

What about words outside the vocabulary? What is their probability?

## Sampling shows what the model learned

Generate one token at a time from the model's predicted distribution. Samples make local preferences and failures easy to inspect.

Sample **incrementally**, one word at a time 

In [ ]:
def sample_once(lm, history, words, rng):
    probs = [lm.probability(word, *history) for word in words]
    return rng.choice(words, p=probs)

In [ ]:
sample_once(baseline, [], sorted(baseline.vocab), np.random.default_rng(7))

In [ ]:
def sample(lm, initial_history, amount_to_sample, seed=7):
    words = sorted(lm.vocab)
    result = list(initial_history)
    rng = np.random.default_rng(seed)
    for _ in range(amount_to_sample):
        history = result[-(lm.order - 1):]
        result.append(sample_once(lm, history, words, rng))
    return result

In [ ]:
sample(baseline, [], 10)

## Evaluating a language model

- **Extrinsic evaluation:** measure whether the model improves a downstream task.
- **Intrinsic evaluation:** give the model held-out text and measure how much probability it assigns to the observed next tokens.

Sequence probability shrinks with length, so we compare average log probability or perplexity.

## Intrinsic evaluation

Give the model text that it did not train on and measure how much probability it assigns to the observed next tokens.

A useful score must account for sequence length, which leads to average log probability and perplexity.

Formalised by

\begin{align}
p(w_1) p(w_2|w_1) \ldots p(w_T|w_1,\ldots,w_{T-1}) &= \prod_{i=1}^T p(w_i|w_1,\ldots,w_{i-1})
\end{align}

But then the longer the sequence, the lower the probability...

$\to$ normalise by the length

### Perplexity 
Given test sequence \\(w_1,\ldots,w_T\\), perplexity \\(\mathrm{PP}\\) is the **geometric mean of inverse next-token probabilities**:

\begin{align}
\mathrm{PP}(w_1,\ldots,w_T) &= \sqrt[T]{\frac{1}{p(w_1)} \frac{1}{p(w_2|w_1)} \ldots} \\
&= \sqrt[T]{\prod_{i=1}^T \frac{1}{p(w_i|w_1,\ldots,w_{i-1})}}
\end{align}

Perplexity for a bigram language model:

\begin{align}
\mathrm{PP}(w_1,\ldots,w_T) &= \sqrt[T]{\prod_{i=1}^T \frac{1}{p(w_i|w_{i-1})}}
\end{align}

Perplexity for a unigram language model:

\begin{align}
\mathrm{PP}(w_1,\ldots,w_T) &= \sqrt[T]{\prod_{i=1}^T \frac{1}{p(w_i)}}
\end{align}

Perplexity for a uniform language model:

\begin{align}
\mathrm{PP}(w_1,\ldots,w_T) &= \sqrt[T]{\prod_{i=1}^T \frac{1}{1/|V|}} = |V|
\end{align}

### Interpreting perplexity

Suppose every position has exactly **2** possible next words, each with probability $\frac{1}{2}$, and the observed word is always one of them.

* $\mathrm{PP}(w_1,\ldots,w_T) = \sqrt[T]{2 \cdot 2  \cdot\ldots} = 2$
* We can read this as roughly two equally plausible choices per token.
* Lower perplexity means the model assigns more probability to the observed sequence.

## Perplexity needs a controlled comparison

Compare models only on the same held-back text with the same tokenisation and vocabulary. A single unseen event with probability zero makes perplexity infinite.

This is why preprocessing and smoothing are part of the model, not bookkeeping.

In [ ]:
perplexity(baseline, test)

Problem: model assigns **zero probability** to words not in the vocabulary. 

In [ ]:
[(w,baseline.probability(w)) for w in test if w not in vocab][:5]

## Sparse counts are unavoidable

Natural-language data contains many rare words and rare contexts. Even a large training corpus leaves plausible n-grams unseen.

Two responses recur throughout the course:

- Share probability across related events, as smoothing does
- Share representations across tokens, as subword models and neural networks do


Let us plot word frequency ranks (x-axis) against frequency (y-axis) 

In [ ]:
plt.xscale('log')
plt.yscale('log') 
plt.plot(ranks, sorted_counts)

In log-space such rank vs frequency graphs are **linear** 

* Known as **Zipf's Law**

Let $r_w$ be the rank of a word \\(w\\), and \\(f_w\\) its frequency:

$$
  f_w \propto \frac{1}{r_w}.
$$

## Fixed vocabulary for word-level models

For the examples below, keep words seen at least twice in the training data and map rarer or unseen words to a single unknown-word symbol. Modern subword tokenisers largely avoid word-level unknowns. We use this fixed mapping only to make word-level perplexity well-defined.

In [13]:
from collections import Counter

train_counts = Counter(train)
lm_vocab = {word for word, count in train_counts.items() if count >= 2}
lm_train = [word if word in lm_vocab else OOV for word in train]
lm_test = replace_OOVs(lm_vocab | {OOV}, test)
lm_baseline = UniformLM(set(lm_train))
round(perplexity(lm_baseline, lm_test), 1)

55.0

## Estimating n-gram probabilities

## Estimating n-gram probabilities

N-gram language models condition on a limited history: 

$$
p(w_i|w_1,\ldots,w_{i-1}) = p(w_i|w_{i-(n-1)},\ldots,w_{i-1}).
$$

What are its parameters (continuous values that control its behaviour)?

One parameter $\theta_{w,h}$ for each word $w$ and history $h=w_{i-(n-1)},\ldots,w_{i-1}$ pair:

$$
p_{\boldsymbol{\theta}}(w|h) = \theta_{w,h}
$$

$p_{\boldsymbol{\theta}}(\text{notebooks}|\text{their}) = \theta_{\text{notebooks, their}}$

### Maximum Likelihood Estimate

Assume training set \\(\mathcal{D}=(w_1,\ldots,w_d)\\)

Find \\(\boldsymbol{\theta}\\) that maximises the log-likelihood of \\(\mathcal{D}\\):

$$
\boldsymbol{\theta}^* = \operatorname*{argmax}_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\mathcal{D})
$$

where

$$
p_{\boldsymbol{\theta}}(\mathcal{D}) = \ldots p_{\boldsymbol{\theta}}(w_i|\ldots w_{i-1}) p_{\boldsymbol{\theta}}(w_{i+1}|\ldots w_i) \ldots 
$$

**Structured Prediction**: this is your continuous optimisation problem!

### Maximum-likelihood estimate

For an n-gram model, maximum likelihood has a direct count-based solution:
$$
p_{\boldsymbol{\theta}^*}(w|h) = \theta^*_{w,h} = \frac{\#_{\mathcal{D}}(h,w)}{\#_{\mathcal{D}}(h)} 
$$

where 

$$
\#_{D}(e) = \text{count of } e \text{ in } D 
$$

Event $h$ means seeing history $h$. Event $(h,w)$ means seeing that history followed by word $w$.

Language-model variants differ in how they estimate and smooth these counts.

## From counts to a unigram model

For a unigram model, maximum likelihood assigns each word its relative frequency:

$$p(w)=\frac{\#(w)}{\sum_{w' \in V} \#(w')}$$

This ignores order, but it gives us a transparent baseline.

What do you think the most probable words are? 

Remember our training set looks like this ...

In [ ]:
lm_train[:12]

In [15]:
unigram = NGramLM(lm_train, 1)
top_probabilities(unigram)

[('.', 0.164),
 ('the', 0.128),
 ('models', 0.064),
 ('students', 0.06),
 ('language', 0.04),
 ('their', 0.032),
 ('contexts', 0.024),
 ('lecture', 0.024)]

The unigram LM has substantially reduced (and hence better) perplexity than the uniform LM:

In [16]:
round(perplexity(lm_baseline, lm_test), 1), round(perplexity(unigram, lm_test), 1)

(55.0, 35.3)

Its samples look (a little) more reasonable:

In [ ]:
print(sample(lm_baseline, [], 10), "\n")
print(sample(unigram, [], 10))

## A bigram adds one word of context

$$p(w_i \mid w_{i-1})=\frac{\#(w_{i-1},w_i)}{\#(w_{i-1})}$$

Samples become more locally fluent, but every unseen transition still receives probability zero.

In [ ]:
bigram = NGramLM(lm_train, 2)
plot_probabilities(bigram, ("the",))

Samples should look (slightly) more fluent:

In [19]:
" ".join(sample(bigram, ["the"], 22, seed=4))

'the students opened their laptops . the students compared language models use attention over tokens . the students studied language models predict words'

How about perplexity?

In [ ]:
perplexity(bigram,lm_test)

Some word--context pairs were not observed in training, hence zero probability...

In [ ]:
bigram.probability("rhyme", "man")

## Smoothing redistributes probability

Maximum-likelihood counts assign too much probability to observed events and none to unseen events.

Smoothing moves some probability from observed n-grams to plausible unseen ones. The important design question is where that probability should go.

### Additive smoothing

Add **pseudo counts** to each event in the dataset 

$$
\theta^{\alpha}_{w,h} = \frac{\#_{\mathcal{D}}(h,w) + \alpha}{\#_{\mathcal{D}}(h) + \alpha \lvert V \rvert} 
$$

## What changes in the next two lectures?

An n-gram stores a separate count for each short context.

- An RNN learns a compact state that summarises a variable-length history.
- A Transformer uses attention to build a contextual representation of every token.

The probability question stays the same. The representation of context changes.

## Summary

- Language models assign probabilities to next tokens and complete sequences.
- N-grams provide a transparent baseline by truncating the context.
- Perplexity supports controlled comparisons when data and tokenisation match.
- Smoothing handles unseen events by sharing probability.
- RNNs and Transformers replace the fixed n-gram context with learned contextual representations.

## Background Reading

- Jurafsky & Martin, [Chapter 3: N-Gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf)
- Jurafsky & Martin, [Appendix C: Kneser-Ney Smoothing](https://web.stanford.edu/~jurafsky/slp3/C.pdf)
- Chen and Goodman (1998), [An Empirical Study of Smoothing Techniques for Language Modeling](https://dash.harvard.edu/bitstream/handle/1/25104739/tr-10-98.pdf?sequence=1)